# WaKi - Movies: Ein intelligentes, stimmungsbasiertes Filmempfehlungssystem

## Einleitung

Klassische Empfehlungssysteme basieren primär auf expliziten Genre-Kategorisierungen (z. B. "Action", "Drama") oder kollaborativen Filterverfahren ("Nutzer, die Film A mochten, sahen auch Film B"). Diese Ansätze stoßen an ihre Grenzen, wenn Nutzer nach Filmen suchen, die einer bestimmten, nuancierten emotionalen Stimmung entsprechen (z. B. *"Ich möchte einen melancholischen, aber gleichzeitig abenteuerlichen und herzerwärmenden Film sehen"*). Die semantische Komplexität solcher Freitext-Anfragen lässt sich durch starre Genre-Tags kaum abbilden.

Das vorliegende Projekt **WaKi-Movies** löst diese Fragestellung durch einen **zwei-stufigen hybriden Retrieval-Ansatz**:
1. **Stimmungsvorhersage (Inferenz):** Ein feinjustiertes Deep-Learning-Modell (basierend auf der Transformer-Architektur `DistilBERT`) übersetzt Freitext-Eingaben in einen kontinuierlichen Wahrscheinlichkeitsvektor über $N = 100$ vordefinierte, latente Stimmungs-Tags (z. B. *melancholic, suspenseful, uplifting*).
2. **Semantischer Vektor-Abgleich (Retrieval):** Dieser generierte Nutzervektor wird mittels mathematischer Ähnlichkeitsmetriken (Kosinus-Ähnlichkeit) mit statischen Stimmungs-Profilvektoren einer Filmdatenbank abgeglichen, um die bestpassenden Filme (Top-K) zu ermitteln.

Die Interaktion erfolgt über einen asynchronen **Telegram-Bot**, der die Benutzerschnittstelle bildet und die Vorhersage- und Retrieval-Pipelines nahtlos verknüpft. Das gesamte System ist containerisiert und für das VPS-Deployment ausgelegt.

## Übersicht
1. [Vorbereitung des Datensatzes](#analysis_cleaning)
2. [Einstiegspunkt und System-Workflow](#starting_point)
3. [Modellentwicklung & Training](#model)
4. [Interaktivität via Telegram](#telegram)

<a id="analysis_cleaning"></a>

## 1. Vorbereitung des Datensatzes

### 1.1 Ursachen für die Reduzierung der Datenmenge auf 13.676 Einträge

Die Reduzierung des ursprünglichen "MovieLens 25M"-Datensatzes (der ca. 62.000 Filme umfasst) auf exakt **13.676 Filme** im bereinigten Datensatz resultiert aus notwendigen Filterungsschritten in unserer Daten-Pipeline:

- **Verfügbarkeit des Tag-Genomes**: Die für das Mood-Training essenziellen Stimmungs-Vektoren (das Tag-Genome) sind im MovieLens-Datensatz nur für knapp 14.000 Filme berechnet worden. Filme ohne diese Vektoren besitzen keine Labels und mussten über einen `inner join` entfernt werden.
- **Ausschluss fehlender Filmbeschreibungen**: Da das Modell lernen soll, aus Freitext Stimmungen vorherzusagen, ist eine Textbeschreibung zwingend erforderlich. Filme, für die aus TMDB keine Beschreibung (`overview`) geladen werden konnte, wurden herausgefiltert.

### 1.2 Eignung der verbleibenden Datenmenge für das Fine-Tuning

Obwohl 13.676 Datensätze für das Training eines neuronalen Netzes von Grund auf unzureichend wären, ist diese menge für unsere Methode optimal:

- **Transfer Learning**: Wir nutzen ein vortrainiertes Transformer-Modell (z. B. DistilBERT), das bereits über umfassendes sprachliches Vorwissen verfügt.
- **Optimaler Bereich (Sweet Spot)**: Für das Fine-Tuning eines solchen Modells gilt eine dichte, qualitativ hochwertige Datenbasis von 10.000 bis 20.000 Beispielen als ideal. Sie bietet ausreichend Varianz für robustes Lernen bei gleichzeitig sehr kurzen Trainingszeiten und minimalem Rauschen.

### 1.3 Datenaufbereitung, Pivotierung und Hugging Face Integration

Die Datenaufbereitung führt Informationen aus MovieLens (Links und Genome-Scores) und TMDB (Film-Metadaten) zusammen. Um Rauschen zu minimieren, werden nur die Top 100 relevantesten Stimmungs-Tags ausgewählt. Ein Tag gilt als aktiv für einen Film, wenn sein Relevanzwert $\ge 0.5$ beträgt.

Folgender Ausschnitt aus [prepare_data.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/src/data/prepare_data.py) zeigt den Kern dieser Pivotierungs- und Filterlogik:

```python
# Auszug aus src/data/prepare_data.py zur Filterung und Pivotierung der Stimmungsdaten

# Selektion der 100 relevantesten Tags basierend auf dem Entscheidungsschwellenwert (Decision Threshold)
top_tag_ids = (
    scores_df[scores_df["relevance"] >= Settings.DECISION_THRESHOLD]["tagId"]
    .value_counts()
    .head(100)
    .index
)
scores_filtered = scores_df[scores_df["tagId"].isin(top_tag_ids)]
scores_named = pd.merge(scores_filtered, tags_df, on="tagId")

# Pivotierung der Daten zur Erzeugung der One-Hot-ähnlichen Multi-Label-Matrix
matrix_df = scores_named.pivot(
    index="movieId", columns="tag", values="relevance"
)
# Binarisierung: Relevanzwerte >= 0.5 werden als 1.0 (zutreffend), sonst 0.0 (nicht zutreffend) gesetzt
matrix_df = (matrix_df >= Settings.DECISION_THRESHOLD).astype(float)
```

<a id="starting_point"></a>

# 2. Einstiegspunkt und System-Workflow

### 2.1 Systemarchitektur und Ausführungssteuerung

Die zentrale Steuerung des Programms erfolgt über die Datei [main.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/main.py). Dieser Einstiegspunkt übernimmt folgende Aufgaben:
1. **Konsistenzprüfung & Datenvorbereitung**: Verifikation der lokalen Existenz des bereinigten Datensatzes (`cleaned_movie_data.csv`) und der Stimmungs-Tag-Definition (`mood_tags.json`). Falls nicht vorhanden, erfolgt ein asynchroner Download und Merge via `create_local_data_file()`.
2. **Modell-Retrieval**: Überprüfung, ob das fertig trainierte Modell lokal vorliegt. Falls nicht, wird es automatisch vom Hugging Face Model Hub geladen (unter Verwendung von `download_model_from_hugging_face`).
3. **Komponenten-Initialisierung**: Instanziierung des `MoodPredictor` (Inferenz-Engine) und des `MovieRecommender` (Retrieval-Engine).
4. **Bot-Start**: Initialisierung und Ausführung des `TelegramBot` im Hintergrund.

Die Ausführungslogik ist in folgendem Codeblock abgebildet:

```python
# Auszug aus main.py: Orchestrierung und Bootstrapping des Gesamtsystems

# Automatisches Laden lokaler Daten oder asynchrones Erzeugen
if not local_csv_file.exists() or not mood_tags_json_file.exists():
    print("Preparing local data...")
    asyncio.run(create_local_data_file())

# Download des fertig trainierten Modells vom Hugging Face Hub bei Bedarf
if not model_path.exists():
    download_model_from_hugging_face(
        repo_id=Settings.HF_MODEL_ID, local_dir=model_path
    )

# Initialisierung der Vorhersage- und Empfehlungsmodule
predictor = MoodPredictor()
recommender = MovieRecommender(data_path=local_csv_file)

# Start des Telegram-Bots
bot = TelegramBot(predictor=predictor, recommender=recommender)
bot.start()
print("Bot started.")
bot.wait()
```

### 2.2 Datenfluss & Sequenzdiagramm

Das Zusammenspiel der Systemkomponenten bei einer Nutzeranfrage lässt sich durch folgendes Sequenzdiagramm visualisieren:

```mermaid
sequenceDiagram
    autonumber
    actor User as Telegram Nutzer
    participant Bot as TelegramBot (bot.py)
    participant Predictor as MoodPredictor (inference.py)
    participant Recommender as MovieRecommender (recommender.py)
    participant Data as Filmdatenbank (cleaned_movie_data.csv)

    User->>Bot: Sendet Freitext (z.B. "düsterer Sci-Fi Thriller, spannend")
    Bot->>Predictor: predict(text)
    Note over Predictor: Tokenisierung & Inferenz<br/>(DistilBERT + Sigmoid Head)
    Predictor-->>Bot: Liefert Wahrscheinlichkeitsverteilung u (Dim 100)
    Bot->>Recommender: get_recommendations(u, top_k=3)
    Note over Recommender: Berechne Kosinus-Ähnlichkeit<br/>zwischen u und Film-Matrix m
    Recommender->>Data: Vergleiche mit gespeicherten Vektoren
    Recommender-->>Bot: Gibt Top 3 Empfehlungen zurück
    Bot->>User: Präsentiert formatierte Filmliste mit Match-%
```

*Hinweis:* Das fertig trainierte Modell wird auf dem Hugging Face Model Hub bereitgestellt (entsprechend der Konfiguration in `Settings.HF_MODEL_ID`).

<a id="model"></a>

# 3. Modellentwicklung und Trainings-Pipeline

### 3.1 Modell-Architektur für Multi-Label-Klassifikation

Da ein Film mehrere Stimmungen gleichzeitig besitzen kann (z. B. sowohl *spannend* als auch *düster*), handelt es sich hierbei um eine **Multi-Label-Klassifikation**. Als Basis-Modell dient `distilbert-base-uncased` (bzw. konfigurierbar in `Settings.MODEL_NAME`).

- **Output Layer**: Der Klassifikationskopf projiziert den Hidden-State des ersten Tokens (`[CLS]`) auf $N = 100$ Output-Logits.
- **Aktivierungsfunktion**: Jedes Ausgangs-Logit $x_i$ wird durch eine **Sigmoid-Funktion** $\sigma(x_i) = \frac{1}{1 + e^{-x_i}}$ unabhängig voneinander auf den Bereich $[0, 1]$ skaliert, um Wahrscheinlichkeiten für jedes Tag zu erhalten.
- **Loss-Funktion**: Trainiert wird das Modell unter Verwendung der **Binary Cross Entropy mit Logits** (`BCEWithLogitsLoss`). Der Gesamtverlust berechnet sich als Mittelwert über alle Labels:
  $$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \cdot \log(\sigma(x_i)) + (1 - y_i) \cdot \log(1 - \sigma(x_i)) \right]$$
  
### 3.2 Tokenisierung und Datenvorbereitung

Die Transformation der Film-Overviews in Token-IDs erfolgt über die Transformers-Pipeline. Dabei werden die Texte auf eine feste Sequenzlänge (`MAX_SEQUENCE_LENGTH = 512`) gebracht. Ein Auszug aus [dataset.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/src/training/dataset.py) zeigt die Tokenisierung und Formatierung der Trainings-Labels:

```python
# Auszug aus src/training/dataset.py: Tokenisierung und Formatierung der Daten

def tokenize_and_format_fn(examples):
    # Tokenisierung des Textes mit Padding und Truncation
    tokenizer_out = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )
    tokenized = dict(tokenizer_out)

    # Konvertierung der Labels in Float-Listen für den BCE Loss
    labels_list = []
    for label in examples["labels"]:
        if isinstance(label, str):
            labels_list.append(json.loads(label))
        else:
            labels_list.append([float(val) for val in label])

    tokenized["labels"] = labels_list
    return tokenized
```

### 3.3 Evaluierungsmetriken & Hyperparameter-Tuning

Für die Leistungsbewertung des Modells werden der **Macro F1-Score** sowie der **ROC-AUC-Wert** verwendet. Der Klassifikationsschwellenwert wird auf $0.5$ festgelegt. Folgender Auszug zeigt die Berechnung der Metriken in [train_model.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/src/training/train_model.py):

```python
# Auszug aus src/training/train_model.py: Berechnung der Evaluierungsmetriken

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids

    # Sigmoid-Aktivierung auf Logits
    probs = 1 / (1 + np.exp(-logits))
    # Binarisierung anhand des Schwellenwerts (Threshold = 0.5)
    predictions = (probs >= Settings.DECISION_THRESHOLD).astype(float)

    # Berechnung des ungewichteten F1-Makro-Scores
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)

    try:
        roc_auc = roc_auc_score(labels, probs, average="macro", multi_class="ovr")
    except Exception as e:
        print(f"Warnung bei ROC-AUC Berechnung: {e}")
        roc_auc = 0.0

    return {"macro_f1": float(macro_f1), "roc_auc": float(roc_auc)}
```

### 3.4 Durchführung des Trainings

Das Modell kann lokal mit folgendem Befehl trainiert werden:
```bash
uv run python -m src.training.train_model --epochs 12 --batch_size 24
```
Für ein schnelles Testen steht ein Dry-Run-Modus zur Verfügung, der nur einen minimalen Datensatz verwendet:
```bash
uv run python -m src.training.train_model --dry_run
```

Zur Optimierung der Trainingsergebnisse wurde ein systematisches Hyperparameter-Tuning über verschiedene Epochen und Batch-Größen hinweg mithilfe des Skripts [run_experiments.sh](file:///Users/sebastianwolf/projects/pythonics/waki-movies/run_experiments.sh) durchgeführt, um die beste Kombination für den F1-Makro-Score zu finden.

<a id="telegram"></a>

# 4. Interaktivität und Empfehlung via Telegram

### 4.1 Asynchrone Anbindung und Empfehlungslogik

Die Interaktion mit dem System geschieht in Echtzeit über Telegram. Um zu verhindern, dass rechenintensive Inferenzschritte (auf der CPU oder GPU) den asynchronen Event-Loop des Bots blockieren, wird der Vorhersage- und Retrievalprozess über `asyncio.to_thread` in einen separaten Thread ausgelagert.

- **Inferenz**: Der Text des Nutzers wird durch den `MoodPredictor` analysiert, welcher eine sortierte Liste aller 100 Stimmungen und deren Wahrscheinlichkeiten liefert.
- **Vektormatching (Kosinus-Ähnlichkeit)**: Der `MovieRecommender` vergleicht diesen vorhergesagten Vektor $ec{u}$ mit den Zeilen der Filmmatrix $M \in \mathbb{R}^{13676 \times 100}$:
  $$\text{Similarity}(ec{u}, ec{m}_j) = \frac{ec{u} \cdot ec{m}_j}{\|\vec{u}\| \|\vec{m}_j\|}$$
  Der Grad der Übereinstimmung wird in Prozent umgerechnet und dem Nutzer angezeigt.

Auszug aus [bot.py](file:///Users/sebastianwolf/projects/pythonics/waki-movies/src/telegram_bot/bot.py):

```python
# Auszug aus src/telegram_bot/bot.py: Asynchroner Message-Handler

async def message_handler(self, update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    if update.message is None or update.message.text is None:
        return

    user_message = update.message.text

    try:
        # Auslagerung der CPU/GPU-intensiven Inferenz in einen separaten Thread
        async with self.recommendation_lock:
            recommendations = await asyncio.to_thread(
                self.get_recommendations_for_message,
                user_message,
            )

        # Senden der formatierten Ergebnisse an den Nutzer
        await update.message.reply_text(self.format_recommendations(recommendations))
    except Exception:
        logger.exception("Failed to create movie recommendations.")
        await update.message.reply_text(
            "Sorry, I couldn't create recommendations right now. Please try again later."
        )
```

### 4.2 Empfehlungsprozess im Recommender

Die eigentliche Berechnung der Kosinus-Ähnlichkeit im `MovieRecommender` sieht wie folgt aus. Hierbei wird der prognostizierte Stimmungs-Wahrscheinlichkeitsvektor des Nutzers mit der vorab berechneten Filmmatrix abgeglichen:

```python
# Auszug aus src/inference/recommender.py: Kosinus-Ähnlichkeit Berechnung

# Vektor des Nutzers aufbauen
user_vector = np.array(
    [user_vector_dict.get(tag, 0.0) for tag in self.mood_tags], dtype=np.float32
)
user_vector_2d = user_vector.reshape(1, -1)

# Berechnung der paarweisen Kosinus-Ähnlichkeit gegen alle Filme in der Matrix
raw_similarities = cosine_similarity(user_vector_2d, self.movie_matrix)[0]
similarities = raw_similarities.astype(np.float32)

# Sortierung und Auswahl der Top-K (z.B. Top-3) Indizes
top_indices = np.argsort(similarities)[::-1][:top_k]
```